# Dual-Style HHT motion transfer

Replace the content motion and the style motion with your own HumanML-263 files to run the same dual-branch transfer. The result is 22 joint positions, shown side by side as content, style, and the transferred motion.

The input is a raw HumanML3D feature: one `.npy` of shape `[T, 263]` at 20 Hz. Do not normalize it. The paper protocol uses global style scale **2.5** and fine-style scale **1.5**.

## Your own motion

If you already have this feature, change the content and style paths in the example cell.

SMPL motion, stored as axis-angle poses and root translation (`poses` and `trans` in an `.npz`), is converted by the HumanML3D scripts in this order: [raw_pose_processing.ipynb](https://github.com/EricGuo5513/HumanML3D/blob/main/raw_pose_processing.ipynb), then [motion_representation.ipynb](https://github.com/EricGuo5513/HumanML3D/blob/main/motion_representation.ipynb). Use the `new_joint_vecs` array they write. Do not apply HumanML3D's own mean and standard deviation. This demo normalizes with the statistics shipped with the released weights.

A motion-capture BVH file is not read directly. Retarget it to an SMPL skeleton and export an SMPL `.npz` with the dataset pipeline in [tempo-changing-music2motion](https://github.com/dongran/tempo-changing-music2motion) (`dataset/`: prepare the BVH, retarget in MotionBuilder, then finalize to NPZ). Then run the two HumanML3D notebooks above.

On Colab, use a GPU runtime (Runtime → Change runtime type → T4 GPU). A free T4 has 16 GB, which is enough for one transfer. The setup cell downloads the code from GitHub and the released weights from Hugging Face. You do not need to upload the checkpoints yourself.

Project page: https://www.dr-lab.org/projects/kinematic-frequency-motion/

In [ ]:
import hashlib
import os
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

WEIGHT_REPO = "randong/kinematic-frequency-motion-weights"
REPO_URL = "https://github.com/dongran/kinematic-frequency-motion.git"
WEIGHTS = [
    "checkpoints/contact_timing.pt",
    "checkpoints/imf_extractor.pt",
    "checkpoints/motionclip_checkpoint/motionclip.pth.tar",
    "checkpoints/motion_vae.ckpt",
    "checkpoints/dual_style_denoiser.ckpt",
]

def find_local_repo():
    here = Path.cwd().resolve()
    for candidate in [here, here.parent, Path("/content/kinematic-frequency-motion")]:
        if (candidate / "demo_dual_style.py").is_file():
            return candidate
    return None

if IN_COLAB:
    import torch
    print('cuda:', torch.cuda.is_available())
    if not torch.cuda.is_available():
        raise RuntimeError('Select Runtime, Change runtime type, T4 GPU, then rerun this cell.')
    print('gpu:', torch.cuda.get_device_name(0))
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "omegaconf", "einops", "timm", "ftfy", "regex", "scipy", "pyyaml",
        "pytorch-lightning==1.8.6", "torchmetrics==0.7", "diffusers", "smplx==0.1.28", "chumpy",
        "huggingface_hub", "git+https://github.com/openai/CLIP.git",
    ])

REPO = find_local_repo()
if REPO is None:
    if not IN_COLAB:
        raise RuntimeError("Run this notebook from the repository root, or open it in Colab.")
    dest = Path("/content/kinematic-frequency-motion")
    subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(dest)])
    REPO = dest
elif IN_COLAB:
    import subprocess
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only"])

from huggingface_hub import hf_hub_download

sums_path = REPO / "checkpoints" / "SHA256SUMS"
if not sums_path.is_file():
    hf_hub_download(WEIGHT_REPO, "SHA256SUMS", local_dir=str(REPO / "checkpoints"))
expected = {}
for line in sums_path.read_text(encoding="utf-8").splitlines():
    parts = line.split()
    if len(parts) < 3:
        continue
    expected[parts[2]] = (parts[0], int(parts[1]))

for rel in WEIGHTS:
    dest = REPO / rel
    name = rel.split("checkpoints/", 1)[-1]
    digest, size = expected[name]
    if dest.is_file() and dest.stat().st_size == size:
        print("already present:", dest.name)
        continue
    hf_hub_download(WEIGHT_REPO, name, local_dir=str(REPO / "checkpoints"))
    got = hashlib.sha256(dest.read_bytes()).hexdigest()
    if got != digest:
        raise RuntimeError(f"checksum mismatch for {name}")
    print("verified", name)

sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "notebooks"))
for rel in ("datasets/__init__.py", "datasets/utils/__init__.py", "datasets/utils/common/__init__.py"):
    init_path = REPO / rel
    init_path.parent.mkdir(parents=True, exist_ok=True)
    init_path.touch()
for key in list(sys.modules):
    if key == "datasets" or key.startswith("datasets."):
        del sys.modules[key]
print("repo:", REPO)


## Examples

Each GIF is rendered in this runtime on the free GPU. The notebook runs the released model here. It does not display a video copied from another machine.

Coarse scale is the global style strength. Fine scale is the local detail strength.

- Turning walk × dance kick: coarse 2.5, fine 10
- Turning walk, paper protocol: coarse 2.5, fine 1.5
- Stair walk × energetic style: coarse 2.5, fine 0, 1, and 3

The rendered result uses the paper's foot-contact cleanup by default; set `FOOT_FIX = False` to show the raw joints.


In [ ]:
from pathlib import Path

stair_root = REPO / "data" / "examples" / "stair"
for rel in ("content/010511.npy", "style/multiple_actions.npy"):
    path = stair_root / rel
    if not path.is_file():
        raise FileNotFoundError(path)

CASES = [
    ("dance_walk", "data/examples/turn_footwork/content", "data/examples/turn_footwork/style", 2.5, 10.0, "Turning walk x dance kick. Coarse 2.5, fine 10."),
    ("turn_protocol", "data/examples/turn_footwork/content", "data/examples/turn_footwork/style", 2.5, 1.5, "Turning walk, paper protocol. Coarse 2.5, fine 1.5."),
    ("stair_fine0", "data/examples/stair/content", "data/examples/stair/style", 2.5, 0.0, "Stair walk x energetic style. Coarse 2.5, fine 0."),
    ("stair_fine1", "data/examples/stair/content", "data/examples/stair/style", 2.5, 1.0, "Stair walk x energetic style. Coarse 2.5, fine 1."),
    ("stair_fine3", "data/examples/stair/content", "data/examples/stair/style", 2.5, 3.0, "Stair walk x energetic style. Coarse 2.5, fine 3."),
]
SEED = 0
FOOT_FIX = True
for name, content, style, scale, fine, note in CASES:
    print(f"{name}: {note}")


## Load the released model

The first run also downloads OpenAI CLIP ViT-B/32 if it is not cached. Loading the checkpoints takes a few minutes.

In [ ]:
import transfer_runtime

if "MODEL" not in globals():
    MODEL = transfer_runtime.load_model(REPO)
print("model ready")


In [ ]:
from IPython.display import Image, Markdown, display
from pathlib import Path
import torch
import transfer_runtime

repo = REPO
if not torch.cuda.is_available():
    raise RuntimeError('This notebook needs a GPU runtime.')
if 'MODEL' not in globals():
    MODEL = transfer_runtime.load_model(repo)

for name, content, style, scale, fine, note in CASES:
    display(Markdown(f"## {note}"))
    report = transfer_runtime.run_pair(
        MODEL,
        repo / content,
        repo / style,
        repo / 'outputs' / 'notebook_demo' / name,
        scale=scale,
        fine_scale=fine,
        seed=SEED,
        foot_fix=FOOT_FIX,
    )
    display(Image(filename=str(report['paths']['gif'])))


## What you should see

Each result is one looping stick-figure GIF: content, style, and the transfer. The body should stay upright, about 1.5 m tall, and the transfer should keep the content route.
